In [2]:
import os, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_curve, f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score, average_precision_score, classification_report
)
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)

set_seed(42)

OKRU_CSV   = "okru_normalised.csv"
PIKABU_CSV = "pikabu2ch_normalised.csv"

MODEL_NAME = "cointegrated/rubert-tiny2"
OUT_DIR    = "models/rubert_tiny_okru"
MAX_LEN    = 256
LR         = 2e-5
EPOCHS     = 4
TRAIN_BS   = 32
EVAL_BS    = 64
GRAD_ACC   = 2


In [4]:
okru = pd.read_csv(OKRU_CSV)
pikabu = pd.read_csv(PIKABU_CSV)

ok_train, ok_val = train_test_split(
    okru, test_size=0.1, random_state=42, stratify=okru["label"]
)

# небольшой dev
pi_dev, pi_test = train_test_split(
    pikabu, test_size=0.8, random_state=42, stratify=pikabu["label"]
)

print(
    f"okru train={len(ok_train)} val={len(ok_val)} | "
    f"pikabu dev={len(pi_dev)} test={len(pi_test)}"
)


okru train=222763 val=24752 | pikabu dev=2839 test=11359


In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def tok_fn(batch):
    return tok(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

ds = {
    "train": Dataset.from_pandas(ok_train[["text","label"]], preserve_index=False).map(tok_fn, batched=True),
    "val":   Dataset.from_pandas(ok_val[["text","label"]],   preserve_index=False).map(tok_fn, batched=True),
    "pi_dev": Dataset.from_pandas(pi_dev[["text","label"]],  preserve_index=False).map(tok_fn, batched=True),
    "pi_test": Dataset.from_pandas(pi_test[["text","label"]],preserve_index=False).map(tok_fn, batched=True),
}
for k in ds:
    ds[k].set_format(type="torch", columns=["input_ids","attention_mask","label"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/222763 [00:00<?, ? examples/s]

Map:   0%|          | 0/24752 [00:00<?, ? examples/s]

Map:   0%|          | 0/2839 [00:00<?, ? examples/s]

Map:   0%|          | 0/11359 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACC,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    report_to="none",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    p1 = (e / e.sum(axis=1, keepdims=True))[:, 1]
    yhat = (p1 >= 0.5).astype(int)
    return {
        "f1": f1_score(labels, yhat),
        "precision": precision_score(labels, yhat),
        "recall": recall_score(labels, yhat),
        "roc_auc": roc_auc_score(labels, p1),
        "pr_auc": average_precision_score(labels, p1),
        "accuracy": accuracy_score(labels, yhat),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    tokenizer=tok,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-334138833.py:40: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [8]:
trainer.train()


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Roc Auc,Pr Auc,Accuracy
1,0.088500,0.078634,0.921418,0.912393,0.930624,0.992927,0.976537,0.971437
2,0.070100,0.072965,0.928282,0.905963,0.951729,0.995164,0.982525,0.973537
3,0.050000,0.066932,0.937353,0.935780,0.938931,0.995620,0.984065,0.977416
4,0.042200,0.069144,0.936275,0.929235,0.943422,0.995632,0.984234,0.976891


TrainOutput(global_step=13924, training_loss=0.08844622674258198, metrics={'train_runtime': 2914.6176, 'train_samples_per_second': 305.718, 'train_steps_per_second': 4.777, 'total_flos': 3285400937189376.0, 'train_loss': 0.08844622674258198, 'epoch': 4.0})

In [9]:
def logits_to_p1(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return (e / e.sum(axis=1, keepdims=True))[:, 1]

val_pred = trainer.predict(ds["val"])
val_p = logits_to_p1(val_pred.predictions)
val_y = val_pred.label_ids

prec, rec, thr = precision_recall_curve(val_y, val_p)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1s))
THR_OKRU = float(thr[best_idx])

yhat = (val_p >= THR_OKRU).astype(int)
print(f"[VAL(okru)] thr={THR_OKRU:.6f}  F1={f1_score(val_y, yhat):.4f} "
      f"P={precision_score(val_y, yhat):.4f} R={recall_score(val_y, yhat):.4f} "
      f"ROC-AUC={roc_auc_score(val_y, val_p):.4f} PR-AUC={average_precision_score(val_y, val_p):.4f}")
print("== VAL report ==\n", classification_report(val_y, yhat, digits=4))


[VAL(okru)] thr=0.498408  F1=0.9376 P=0.9356 R=0.9396 ROC-AUC=0.9956 PR-AUC=0.9841
== VAL report ==
               precision    recall  f1-score   support

           0     0.9867    0.9858    0.9863     20298
           1     0.9356    0.9396    0.9376      4454

    accuracy                         0.9775     24752
   macro avg     0.9612    0.9627    0.9619     24752
weighted avg     0.9775    0.9775    0.9775     24752



In [10]:
pi_dev_pred  = trainer.predict(ds["pi_dev"])
pi_test_pred = trainer.predict(ds["pi_test"])

pi_dev_p  = logits_to_p1(pi_dev_pred.predictions)
pi_test_p = logits_to_p1(pi_test_pred.predictions)
pi_dev_y  = pi_dev_pred.label_ids
pi_test_y = pi_test_pred.label_ids

# порог под pikabu
prec, rec, thr = precision_recall_curve(pi_dev_y, pi_dev_p)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1s))
THR_PIKABU = float(thr[best_idx])

# отчёт на тесте pikabu с доменным порогом
test_yhat = (pi_test_p >= THR_PIKABU).astype(int)
print(f"[PIKABU] tuned_thr={THR_PIKABU:.6f}")
print("== TEST(pikabu) report ==\n", classification_report(pi_test_y, test_yhat, digits=4))
print(f"[PIKABU TEST] ROC-AUC={roc_auc_score(pi_test_y, pi_test_p):.4f}  PR-AUC={average_precision_score(pi_test_y, pi_test_p):.4f}")


[PIKABU] tuned_thr=0.001828
== TEST(pikabu) report ==
               precision    recall  f1-score   support

           0     0.9165    0.8469    0.8803      7564
           1     0.7350    0.8461    0.7866      3795

    accuracy                         0.8466     11359
   macro avg     0.8257    0.8465    0.8335     11359
weighted avg     0.8558    0.8466    0.8490     11359

[PIKABU TEST] ROC-AUC=0.9157  PR-AUC=0.8453


In [18]:
import torch
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, precision_score, recall_score, \
                            accuracy_score, roc_auc_score, average_precision_score, classification_report

def fit_temperature_on_val(logits_np, labels_np, max_iters=500, lr=0.01):
    logits = torch.tensor(logits_np, dtype=torch.float32)
    labels = torch.tensor(labels_np, dtype=torch.long)
    T = torch.nn.Parameter(torch.ones(1) * 1.0)
    opt = torch.optim.LBFGS([T], lr=1.0, max_iter=50)
    loss_fn = torch.nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        scaled = logits / T.clamp(min=1e-3, max=100.0)
        loss = loss_fn(scaled, labels)
        loss.backward()
        return loss

    opt.step(closure)
    opt2 = torch.optim.Adam([T], lr=lr)
    for _ in range(max_iters):
        opt2.zero_grad()
        scaled = logits / T.clamp(min=1e-3, max=100.0)
        loss = loss_fn(scaled, labels)
        loss.backward()
        opt2.step()

    return float(T.detach().cpu().clamp(min=1e-3, max=100.0).item())

def p1_from_logits(logits, T=1.0):
    scaled = logits / T
    e = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return (e / e.sum(axis=1, keepdims=True))[:, 1]
T_val = fit_temperature_on_val(val_pred.predictions, val_pred.label_ids)
print(f"[CAL] Optimal temperature on okru-val: T = {T_val:.3f}")
val_p_cal = p1_from_logits(val_pred.predictions, T=T_val)
val_y     = val_pred.label_ids
prec, rec, thr = precision_recall_curve(val_y, val_p_cal)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.nanargmax(f1s))
THR_GLOBAL = float(thr[best_idx])

yhat_val = (val_p_cal >= THR_GLOBAL).astype(int)
print(f"[okru-val CAL] thr={THR_GLOBAL:.4f}  "
      f"F1={f1_score(val_y, yhat_val):.4f}  P={precision_score(val_y, yhat_val):.4f}  "
      f"R={recall_score(val_y, yhat_val):.4f}  AUC={roc_auc_score(val_y, val_p_cal):.4f}  "
      f"PR-AUC={average_precision_score(val_y, val_p_cal):.4f}")

pi_test_p_cal = p1_from_logits(pi_test_pred.predictions, T=T_val)
pi_test_y     = pi_test_pred.label_ids
yhat_test     = (pi_test_p_cal >= THR_GLOBAL).astype(int)

print("== TEST(pikabu) with global calibrated threshold ==")
print(classification_report(pi_test_y, yhat_test, digits=4))
print(f"[PIKABU TEST CAL] ROC-AUC={roc_auc_score(pi_test_y, pi_test_p_cal):.4f}  "
      f"PR-AUC={average_precision_score(pi_test_y, pi_test_p_cal):.4f}")


[CAL] Optimal temperature on okru-val: T = 1.272
[okru-val CAL] thr=0.4987  F1=0.9376  P=0.9356  R=0.9396  AUC=0.9956  PR-AUC=0.9841
== TEST(pikabu) with global calibrated threshold ==
              precision    recall  f1-score   support

           0     0.7977    0.9603    0.8715      7564
           1     0.8668    0.5146    0.6458      3795

    accuracy                         0.8114     11359
   macro avg     0.8323    0.7375    0.7587     11359
weighted avg     0.8208    0.8114    0.7961     11359

[PIKABU TEST CAL] ROC-AUC=0.9157  PR-AUC=0.8453


In [20]:
# сохраняем модель
import os, json

SAVE_DIR = "final_model/rubert-tiny"
os.makedirs(SAVE_DIR, exist_ok=True)

trainer.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)

infer_cfg = {
    "model_name": MODEL_NAME,
    "max_length": int(MAX_LEN),
    "temperature": float(T_val),
    "threshold_global": float(THR_GLOBAL),
    "label_mapping": {"non_toxic": 0, "toxic": 1}
}
with open(os.path.join(SAVE_DIR, "inference_config.json"), "w", encoding="utf-8") as f:
    json.dump(infer_cfg, f, ensure_ascii=False, indent=2)

print(f"Saved to: {SAVE_DIR}")


Saved to: final_model/rubert-tiny


In [21]:
# качаем архив
!zip -r /content/rubert_tiny_model.zip /content/final_model/rubert-tiny

from google.colab import files
files.download('/content/rubert_tiny_model.zip')


  adding: content/final_model/rubert-tiny/ (stored 0%)
  adding: content/final_model/rubert-tiny/config.json (deflated 49%)
  adding: content/final_model/rubert-tiny/model.safetensors (deflated 8%)
  adding: content/final_model/rubert-tiny/training_args.bin (deflated 53%)
  adding: content/final_model/rubert-tiny/special_tokens_map.json (deflated 80%)
  adding: content/final_model/rubert-tiny/vocab.txt (deflated 64%)
  adding: content/final_model/rubert-tiny/tokenizer_config.json (deflated 73%)
  adding: content/final_model/rubert-tiny/tokenizer.json (deflated 73%)
  adding: content/final_model/rubert-tiny/inference_config.json (deflated 26%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>